prepare_data.py

In [ ]:
from transformers import RobertaTokenizerFast
from datasets import load_datasets, concatenate_dataset, load_from_disk

text_split_pct = 0.005
context_length = 512
path_to_data_store = "/mnt/datadrive/data/prepped_data/roberta_data"
huggingface_cache_dir = "/mnt/datadrive/data/huggingface_cache"
seed = 42
num_workers = 32
hf_model_name = "FacebookAI/roberta-base"

def prepare_data():

    tokenizer = RobertaTokenizerFast.from_pretrained(hf_model_name)

    wikidataset = load_dataset("wikipedia", "20220301.en", cache_dir=huggingface_cache_dir)
    bookdataset = load_dataset("bookcorpus/bookcorpus", cache_dir=huggingface_cache_dir)

    wikidataset = wikidataset.select_columns(["text"])
    bookdataset = bookdataset.select_columns(["text"])

    dataset = concatenate_dataset([wikidataset["train"], bookdataset["train"]])

    dataset = dataset.train_test_split(test_size=text_split_pct, seed=seed)


    def compute_tokens(examples):
        sentences = []
        for example in examples["text"]:
            sentences.append(" ".join(example.replace("\n", "").replace("\t", "").split()))
        tokenized = tokenizer(sentences, return_attention_mask=False)
        return tokenized


    tokenized_data = dataset.map(
        compute_tokens,
        batched=True,
        num_proc=num_workers,
        remove_columns="text",
    )


    def group_texts(batch):

        concatenated_texts = []

        for sentence in batch["input_ids"]:
            concatenated_texts.extend(sentence)

        concatenated_texts = [
            concatenated_texts[i : i + context_length]
            for i in range(0, len(concatenated_texts), context_length)
        ]

        data = {"input_ids": concatenated_texts}

        return data

    tokenized_data.map(
        group_texts,
        batched=True,
        num_proc=num_workers
    )

    tokenized_data.save_to_disk(path_to_data_store)

utils.py

In [ ]:
import torch
import random
from typing import Literal
from transformers import RobertaTokenizerFast
from dataclasses import dataclass, asdict


@dataclass
class RobertaConfig:
    vocab_size: int = 50265
    start_token: int = 0
    end_token: int = 2
    pad_token: int = 2
    mask_token: int = 50264

    embedding_dimension: int = 768
    num_transformer_blocks: int = 12
    num_attention_heads: int = 12
    mlp_ratio: int = 4
    layer_norm_eps: float = 1e-6
    hidden_dropout_p: float = 0.1
    attention_dropout_p: float = 0.1
    context_length: int = 512

    masking_prob: int = 0.15

    hf_model_name: str = "FacebookAI/roberta-base"

    pretrained_backbone: Literal["pretrained", "pretrained_huggingface", "random"] = "pretrained"
    path_to_pretrained_weights: str = None

    def to_dict(self):
        return asdict(self)


def random_masking_text(tokens,
                        special_tokens_mask,
                        vocab_size=50264,
                        special_ids=(0, 1, 2, 3, 50264),
                        mask_ratio=0.15,
                        mask_token=50264):

    non_special_ids = list(set(range(vocab_size)) - set(special_ids))

    random_masking = torch.rand(*tokens.shape)

    random_masking[special_tokens_mask == 1] = 1

    random_masking = (random_masking < mask_ratio)

    labels = torch.full((tokens.shape), -100)

    labels[random_masking] = tokens[random_masking]

    random_selected_idx = random_masking.nonzero()

    masking_flag = torch.rand(len(random_selected_idx))

    masking_flag = (masking_flag < 0.8)

    selected_index_for_masking = random_selected_idx[masking_flag]

    unselected_idx_for_masking = random_selected_idx[~masking_flag]

    masking_flag = torch.rand(len(unselected_idx_for_masking))
    masking_flag = (masking_flag < 0.5)

    selected_idx_for_random_filling = unselected_idx_for_masking[masking_flag]

    selected_idx_to_be_left_alone = unselected_idx_for_masking[~masking_flag]

    if len(selected_index_for_masking) > 0:
        tokens[selected_index_for_masking[:, 0], selected_index_for_masking[:, 1]] = mask_token

    if len(selected_idx_for_random_filling) > 0:
        randomly_selected_tokens = torch.tensor(random.sample(non_special_ids, len(selected_idx_for_random_filling)))
        tokens[selected_idx_for_random_filling[:, 0], selected_idx_for_random_filling[:, 1]] = randomly_selected_tokens


    return tokens, labels





def RobertaMaskedLMCollateFunction(config):
    tokenizer = RobertaTokenizerFast.from_pretrained(config.hf_model_name)

    def _collate_fn(batch):
        input_ids = [torch.tensor(sample["input_ids"]) for sample in batch]

        attention_mask = [torch.ones(len(sample)) for sample in input_ids]
        attention_mask = torch.nn.utils.rnn.pad_sequence(attention_mask, padding_value=0, batch_first=True)

        input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, padding_value=tokenizer.pad_token_id, batch_first=True)

        special_tokens_mask = torch.isin(input_ids, torch.tensor(tokenizer.all_special_ids))

        tokens, labels = random_masking_text(input_ids,
                                             special_tokens_mask,
                                             vocab_size=tokenizer.vocab_size,
                                             special_ids=tokenizer.all_special_ids,
                                             mask_ratio=config.masking_prob,
                                             mask_token=tokenizer.mask_token_id)


        return {"input_ids": tokens,
                "attention_mask": attention_mask,
                "labels": labels}

    return _collate_fn


model.py

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from safetensors.torch import load_file
from transformers import RobertaModel as HFRobertaModel

In [ ]:
class RobertaEmbeddings(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.word_embeddings = nn.Embedding(
            config.vocab_size, config.embedding_dimension, padding_idx=config.pad_token
        )

        self.position_embeddings = nn.Embedding(config.context_length, config.embedding_dimension)

        self.layer_norm = nn.LayerNorm(config.embedding_dimension, eps=config.layer_norm_eps)
        self.dropout = nn.Dropout(config.hidden_dropout_p)


    def forward(self, input_ids):

        batch_size, seq_len = input_ids.shape

        x = self.word_embeddings(input_ids)

        avail_idx = torch.arange(0, seq_len, dtype=torch.long, device=input_ids.device)
        pos_embed = self.position_embeddings(avail_idx)

        x = x + pos_embed

        x = self.layer_norm(x)
        x = self.dropout(x)

        return x


In [ ]:
class RobertaAttention(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.config = config

        assert config.embedding_dimension % config.num_attention_heads == 0, "embedding_dimension must be divisible by num_attention_heads"
        self.head_dim = config.embedding_dimension // config.num_attention_heads

        self.q_proj = nn.Linear(config.embedding_dimension, config.embedding_dimension)
        self.k_proj = nn.Linear(config.embedding_dimension, config.embedding_dimension)
        self.v_proj = nn.Linear(config.embedding_dimension, config.embedding_dimension)

        self.out_proj = nn.Linear(config.embedding_dimension, config.embedding_dimension)


    def forward(self, x, attention_mask=None):

        batch, seq_len, embed_dim = x.shape

        q = self.q_proj(x).reshape(batch, seq_len, self.config.num_attention_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).reshape(batch, seq_len, self.config.num_attention_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).reshape(batch, seq_len, self.config.num_attention_heads, self.head_dim).transpose(1, 2)

        attention_out = F.scaled_dot_product_attention(
            q, k, v,
            attention_mask=attention_mask,
            dropout_p=self.config.attention_dropout_p if self.training else 0.0
        )

        attention_out = attention_out.tranpose(1, 2).flatten(2)
        attention_out = self.out_proj(attention_out)

        return attention_out


In [ ]:
class RobertaFeedForward(nn.Module):

    def __init__(self, config):
        super().__init__()

        hidden_size = config.embedding_dimension * config.mlp_ratio
        self.intermediate_dense = nn.Linear(config.embedding_dimension, hidden_size)
        self.activation = nn.GELU()
        self.intermediate_dropout = nn.Dropout(config.hidden_dropout_p)

        self.output_dense = nn.Linear(hidden_size, config.embedding_dimension)
        self.output_dropout = nn.Dropout(config.hidden_dropout_p)


    def forward(self, x):

        x = self.intermediate_dense(x)
        x = self.activation(x)
        x = self.intermediate_dropout(x)

        x = self.output_dense(x)
        x = self.output_dropout(x)

        return x



In [ ]:
class  RobertaEncoderLayer(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.attention = RobertaAttention(config)
        self.dropout = nn.Dropout(config.hidden_dropout_p)
        self.layer_norm = nn.LayerNorm(config.embedding_dimension, eps=config.layer_norm_eps)
        self.feed_forward = RobertaFeedForward(config)
        self.final_layer_norm = nn.LayerNorm(config.embedding_dimension, eps=config.layer_norm_eps)

    def forward(self, x, attention_mask=None):

        x = x + self.dropout(self.attention(x, attention_mask=attention_mask))
        x = self.layer_norm(x)

        x = x + self.feed_forward(x)
        x = self.final_layer_norm(x)

        return x

In [ ]:
class RobertaEncoder(nn.Module):

    def __init__(self, config):
        super(RobertaEncoder, self).__init__()

        self.config = config

        self.layers = nn.ModuleList(
            [
                RobertaEncoderLayer(config) for _ in range(self.config.num_transformer_block)
            ]
        )

    def forward(self, x, attention_mask=None):
        batch_size, seq_len, embed_dim = x.shape

        if attention_mask is not None:
            attention_mask = attention_mask.unsqueeze(1).unsqueeze(1).repeat(1, 1, seq_len, 1)

        for layer in self.layers:
            x = layer(x, attention_mask=attention_mask)

        return x


In [ ]:
class RobertaMLHead(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.config = config

        self.dense = nn.Linear(config.embedding_dimension, config.embedding_dimension)
        self.layer_norm = nn.LayerNorm(config.embedding_dimension, eps=config.layer_norm_eps)
        self.activation = nn.GELU()

        self.decoder = nn.Linear(config.embedding_dimension, config.vocab_size)


    def forward(self, inputs):

        x = self.dense(inputs)
        x = self.activation(x)
        x = self.layer_norm(x)

        x = self.decoder(x)

        return x

In [ ]:
class RobertaModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.embeddings = RobertaEmbeddings(config)
        self.encoder = RobertaEncoder(config)

    def forward(self, input_ids, attention_mask):
        embeddings = self.embeddings(input_ids)
        output = self.encoder(embeddings)

        return output

In [ ]:
class RobertaForMaskedLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.roberta = RobertaModel(config)
        self.mlm_head = RobertaMLHead(config)

        self.apply(_init_weights_)

    def forward(self, input_ids, attention_mask=None, labels=None):
        hidden_states = self.roberta(input_ids, attention_mask)
        preds = self.mlm_head(hidden_states)

        loss = None

        if labels is not None:
            # preds:  B x S x N -> B*S x N
            # labels: B x S    -> B*S

            preds = preds.flatten(end_dim=1)
            labels = labels.flatten()

            loss = F.cross_entropy(preds, labels)

            return hidden_states, preds, loss

        return hidden_states, preds



In [ ]:
class RobertaForQuestionAnswering(nn.Module):
    pass

In [ ]:
def _init_weights_(module):

    if isinstance(module, nn.Linear):
        module.weight.data.normal_(mean=0.0, std=0.02)
        if module.bias is not None:
            module.bias.data.zero_()

    elif isinstance(module, nn.Embedding):
        module.weight.data.normal_(mean=0.0, std=0.02)
        if module.padding_idx is not None:
            module.weight.data[module.padding_idx].zero_()

    elif isinstance(module, nn.LayerNorm):
        module.bias.data.zero_()
        module.weight.data.fill_(1.0)



In [ ]:
"""
Pretty straightforward Masked Language Modeling pretraining script! This will require a ton of data
and a couple GPUs though to do in a reasonable amount of time, so if you dont have that, when we move onto
finetuning, you can just load a pretrained backbone from Huggingface instead!
"""
import os
import shutil
import numpy as np
import argparse
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchmetrics import Accuracy

from datasets import load_from_disk
from transformers import RobertaTokenizerFast, set_seed, get_scheduler
from accelerate import Accelerator

from utils import RobertaConfig, RobertaMaskedLMCollateFunction
from model import RobertaForMaskedLM

def parse_args():
    ### PARSE COMMAND LINE ARGS ###
    parser = argparse.ArgumentParser(description="RoBERTa Pretraining Arguments on Wikipedia + BookCorpus")
    parser.add_argument(
        "--experiment_name",
        required=True,
        type=str
    )

    parser.add_argument(
        "--working_directory",
        required=True,
        type=str
    )

    ##########################
    ### HUGGINGFACE CONFIG ###
    ##########################

    parser.add_argument(
        "--hf_model_name",
        help="Huggingface model name we want to use for the tokenizer",
        default="FacebookAI/roberta-base",
        type=str
    )

    #########################
    ### DATASET ARGUMENTS ###
    #########################

    parser.add_argument(
        "--path_to_prepped_data",
        required=True,
        help="Path to data prepared in `prepare_data.py`",
        type=str
    )

    parser.add_argument(
        "--context_length",
        help="Max sequence length we want the model to accept",
        default=512,
        type=int
    )

    parser.add_argument(
        "--masking_probability",
        help="Probability of token to be selected to be masked",
        default=0.15,
        type=float
    )

    parser.add_argument(
        "--num_workers",
        help="Number of workers for dataloading",
        default=24,
        type=int
    )

    #######################
    ### MODEL ARGUMENTS ###
    #######################

    ### WAV2VEC2 FEATURE ENCODER CONVOlUTION CONFIG ###

    parser.add_argument(
        "--hidden_dropout_p",
        help="Dropout probability on all linear layers",
        default=0.1,
        type=float
    )

    ### CONVOLUTIONAL POSITIONAL EMBEDDINGS ###
    parser.add_argument(
        "--attention_dropout_p",
        help="Dropout probability on attention matrix",
        default=0.1,
        type=float
    )

    ### TRANSFORMER CONFIG ###
    parser.add_argument(
        "--num_transformer_blocks",
        help="Number of transformer blocks in model",
        default=12,
        type=int
    )

    parser.add_argument(
        "--num_attention_heads",
        help="Number of heads of attention",
        default=12,
        type=int
    )

    parser.add_argument(
        "--embedding_dimension",
        help="Transformer embedding dimension",
        default=768,
        type=int
    )

    parser.add_argument(
        "--mlp_ratio",
        help="Hidden layer expansion factor for feed forward layers",
        default=4,
        type=int
    )

    parser.add_argument(
        "--layer_norm_eps",
        help="error added to layer norm to avoid divide by zero",
        default=1e-5,
        type=float
    )

    parser.add_argument(
        "--initializer_range",
        help="Standard deviation of linear layers initialized as normal distribution",
        default=0.02,
        type=float
    )

    ##############################
    ### TRAINING CONFIGURATION ###
    ##############################

    parser.add_argument(
        "--per_gpu_batch_size",
        help="Overall batch size per gpu during training",
        default=128,
        type=int
    )

    parser.add_argument(
        "--gradient_accumulation_steps",
        help="Splits per_gpu_batch_size by gradient_accumulation_steps",
        default=4,
        type=int
    )

    parser.add_argument(
        "--num_training_steps",
        help="Number of training steps to take",
        default=250000,
        type=int
    )

    parser.add_argument(
        "--max_grad_norm",
        help="Max gradient norm used for stabilizing training with gradient clipping",
        default=1.0,
        type=float
    )

    parser.add_argument(
        "--num_warmup_steps",
        type=int,
        default=20000,
        help="Number of steps for the warmup in the lr scheduler."
    )

    parser.add_argument(
        "--lr_scheduler_type",
        type=str,
        default="linear",
        help="The scheduler type to use.",
        choices=["linear", "cosine", "cosine_with_restarts", "polynomial", "constant", "constant_with_warmup"],
    )

    parser.add_argument(
        "--logging_steps",
        help="Number of iterations for every log of metrics to wandb",
        default=1,
        type=int
    )

    parser.add_argument(
        "--evaluation_interval",
        help="Number of iterations for every evaluation and plotting",
        default=2500,
        type=int
    )

    parser.add_argument(
        "--checkpoint_interval",
        help="Number of iterations for checkpointing",
        default=2500,
        type=int
    )

    parser.add_argument(
        "--learning_rate",
        help="Max learning rate for all Learning Rate Schedulers",
        default=6e-4,
        type=float
    )

    parser.add_argument(
        "--bias_weight_decay",
        help="Apply weight decay to bias",
        default=False,
        action=argparse.BooleanOptionalAction
    )

    parser.add_argument(
        "--norm_weight_decay",
        help="Apply weight decay to normalization weight and bias",
        default=False,
        action=argparse.BooleanOptionalAction
    )

    parser.add_argument(
        "--weight_decay",
        help="Weight decay constant for AdamW optimizer",
        default=0.01,
        type=float
    )

    parser.add_argument(
        "--adam_beta1",
        type=float,
        default=0.9,
        help="Beta1 for AdamW optimizer",
    )

    parser.add_argument(
        "--adam_beta2",
        type=float,
        default=0.98,
        help="Beta2 for AdamW optimizer",
    )

    parser.add_argument(
        "--adam_epsilon",
        type=float,
        default=1e-6,
        help="Epsilon for AdamW optimizer",
    )

    parser.add_argument(
        "--num_keep_checkpoints",
        help="Number of Checkpoints to Keep, if None, all checkpoints will be saved",
        default=None,
        type=int
    )

    parser.add_argument(
        "--seed",
        help="Set seed in model for reproducible training",
        default=None,
        type=int
    )

    parser.add_argument(
        "--resume_from_checkpoint",
        help="Checkpoint folder for model to resume training from, inside the experiment folder",
        default=None,
        type=str
    )

    #############################
    ### LOGGING CONFIGURATION ###
    #############################

    parser.add_argument(
        "--log_wandb",
        help="Flag to enable logging to wandb",
        default=False,
        action=argparse.BooleanOptionalAction
    )

    args = parser.parse_args()

    return args

### Parse Arguments ###
args = parse_args()

### Set Seed ###
if args.seed is not None:
    set_seed(args.seed)

### Instantiate Accelerate ###
path_to_experiment = os.path.join(args.working_directory, args.experiment_name)
accelerator = Accelerator(project_dir=path_to_experiment,
                          log_with="wandb" if args.log_wandb else None)
if args.log_wandb:
    accelerator.init_trackers(args.experiment_name)

### Define Tokenizer ###
tokenizer = RobertaTokenizerFast.from_pretrained(args.hf_model_name)

### Prepare Config File ###
config = RobertaConfig(

    vocab_size = tokenizer.vocab_size,
    start_token = tokenizer.bos_token_id,
    end_token = tokenizer.eos_token_id,
    pad_token = tokenizer.pad_token_id,
    mask_token = tokenizer.mask_token_id,
    embedding_dimension = args.embedding_dimension,
    num_transformer_blocks = args.num_transformer_blocks,
    num_attention_heads = args.num_attention_heads,
    mlp_ratio = args.mlp_ratio,
    layer_norm_eps = args.layer_norm_eps,
    hidden_dropout_p = args.hidden_dropout_p,
    attention_dropout_p = args.attention_dropout_p,
    context_length = args.context_length,
    masking_prob = args.masking_probability,
    hf_model_name = args.hf_model_name

)

### Load Dataset ###
tokenized_data = load_from_disk(args.path_to_prepped_data)

### Load Model ###
model = RobertaForMaskedLM(config)
model_parameters = filter(lambda p: p.requires_grad, model.parameters())
params = sum([np.prod(p.size()) for p in model_parameters])
accelerator.print("Number of Parameters:", params)

## Define DataLoader ###
collate_fn = RobertaMaskedLMCollateFunction(config)
mini_batchsize = args.per_gpu_batch_size // args.gradient_accumulation_steps

train_dataloader = DataLoader(tokenized_data["train"],
                              batch_size=mini_batchsize,
                              collate_fn=collate_fn,
                              num_workers=args.num_workers,
                              shuffle=True)

eval_dataloader = DataLoader(tokenized_data["test"],
                             batch_size=mini_batchsize,
                             collate_fn=collate_fn,
                             num_workers=args.num_workers,
                             shuffle=False)

### PREPARE OPTIMIZER ###
if (not args.bias_weight_decay) or (not args.norm_weight_decay):
    accelerator.print("Disabling Weight Decay on Some Parameters")
    weight_decay_params = []
    no_weight_decay_params = []
    for name, param in model.named_parameters():

        if param.requires_grad:

            ### Dont have Weight decay on any bias parameter (including norm) ###
            if "bias" in name and not args.bias_weight_decay:
                no_weight_decay_params.append(param)

            ### Dont have Weight Decay on any Norm scales params (weights) ###
            elif "groupnorm" in name and not args.norm_weight_decay:
                no_weight_decay_params.append(param)

            else:
                weight_decay_params.append(param)

    optimizer_group = [
        {"params": weight_decay_params, "weight_decay": args.weight_decay},
        {"params": no_weight_decay_params, "weight_decay": 0.0}
    ]

    optimizer = torch.optim.AdamW(optimizer_group,
                                  lr=args.learning_rate,
                                  betas=[args.adam_beta1, args.adam_beta2],
                                  eps=args.adam_epsilon)

else:
    optimizer = torch.optim.AdamW(model.parameters(),
                                lr=args.learning_rate,
                                betas=[args.adam_beta1, args.adam_beta2],
                                eps=args.adam_epsilon)

### DEFINE SCHEDULER ###
scheduler = get_scheduler(
    name=args.lr_scheduler_type,
    optimizer=optimizer,
    num_warmup_steps=args.num_warmup_steps,
    num_training_steps=args.num_training_steps,
)

### PREPARE EVERYTHING ###
model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader
)
accelerator.register_for_checkpointing(scheduler)

### Define Accuracy ###
accuracy_func = Accuracy(task="multiclass", num_classes=tokenizer.vocab_size, ignore_index=-100).to(accelerator.device)

########################################################
############## TRAINING SCRIPT #########################
########################################################

### RESUME FROM CHECKPOINT ###
if args.resume_from_checkpoint is not None:

    ### Grab path to checkpoint ###
    path_to_checkpoint = os.path.join(path_to_experiment, args.resume_from_checkpoint)

    ### Load checkpoint on main process first, recommended here: (https://huggingface.co/docs/accelerate/en/concept_guides/deferring_execution) ###
    with accelerator.main_process_first():
        accelerator.load_state(path_to_checkpoint)

    ### Start completed steps from checkpoint index ###
    completed_steps = int(args.resume_from_checkpoint.split("_")[-1])
    accelerator.print(f"Resuming from Iteration: {completed_steps}")

else:
    completed_steps = 0

train = True
progress_bar = tqdm(range(completed_steps, args.num_training_steps), disable=not accelerator.is_local_main_process)

while train:

    ### Keep Track of Accumulated Mini-Steps ###
    accumulate_steps = 0

    ### Accumulated Loss ###
    accumulate_loss = 0

    ### Keep Track of Accuracy ###
    accuracy = 0

    for batch in train_dataloader:

        ### Make sure everything is on GPU ###
        batch = {k:v.to(accelerator.device) for (k,v) in batch.items()}

        ### Pass Batch through Model ###
        logits, preds, loss = model(**batch)

        ### Scale Loss by Gradient Accumulation Steps ###
        loss = loss / args.gradient_accumulation_steps
        accumulate_loss += loss

        ### Compute Gradients ###
        accelerator.backward(loss)

        ### Compute TopK Accuracy ###
        labels = batch["labels"].flatten()
        acc = accuracy_func(preds, labels)
        accuracy += acc / args.gradient_accumulation_steps

        ### Iterate Accumulation ###
        accumulate_steps += 1

        if accumulate_steps % args.gradient_accumulation_steps == 0:

            ### Update Model ###
            accelerator.clip_grad_norm_(model.parameters(), args.max_grad_norm)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

            ### Update Scheduler ###
            scheduler.step()

            ### Log Results!! ###
            if completed_steps % args.logging_steps == 0:

                accumulate_loss = accumulate_loss.detach()
                accuracy = accuracy.detach()

                if accelerator.state.num_processes > 1:
                    accumulate_loss = torch.mean(accelerator.gather_for_metrics(accumulate_loss))
                    accuracy = torch.mean(accelerator.gather_for_metrics(accuracy))

                log = {"train_loss": accumulate_loss,
                       "train_acc": accuracy,
                       "learning_rate": scheduler.get_last_lr()[0]}

                logging_string = f"[{completed_steps}/{args.num_training_steps}] Training Loss: {accumulate_loss} | Training Acc: {accuracy}"

                if accelerator.is_main_process:
                    progress_bar.write(logging_string)

                if args.log_wandb:
                    accelerator.log(log, step=completed_steps)

            ### Evaluation Loop ###
            if completed_steps % args.evaluation_interval == 0:
                if accelerator.is_main_process:
                    progress_bar.write("Evaluating Model!!")

                model.eval()

                ### Dictionary to Store Results ###
                log = {"val_loss": 0,
                       "val_acc": 0}

                ### Iterate Data ###
                num_losses = 0
                for batch in tqdm(eval_dataloader, disable=not accelerator.is_main_process):

                    ### Move everything to GPU ###
                    batch = {k:v.to(accelerator.device) for (k,v) in batch.items()}

                    ### Pass through model ###
                    with torch.inference_mode():
                        logits, preds, loss = model(**batch)

                    ### Grab Loss ###
                    loss = loss.detach()
                    if accelerator.num_processes > 1:
                        loss = torch.mean(accelerator.gather_for_metrics(loss))

                    ### Compute Accuracy ###
                    labels = batch["labels"].flatten()
                    accuracy = accuracy_func(preds, labels)

                    ### Add to our Logs ###
                    log["val_loss"] += loss
                    log["val_acc"] += accuracy
                    num_losses += 1

                ### Divide Log by Num Losses ###
                log["val_loss"] = log["val_loss"] / num_losses
                log["val_acc"] = log["val_acc"] / num_losses

                ## Print to Console ###
                logging_string = f"[{completed_steps}/{args.num_training_steps}] Validation Loss: {log["val_loss"]} | Validation Acc: {log["val_acc"]}"

                ### Print out Log ###
                if accelerator.is_main_process:
                    progress_bar.write(logging_string)

                if args.log_wandb:
                    accelerator.log(log, step=completed_steps)

                ### Return Back to Training Mode ###
                model.train()



            ### Checkpoint Model (Only need main process for this) ###
            if (completed_steps % args.checkpoint_interval == 0):

                ### Save Checkpoint ###
                path_to_checkpoint = os.path.join(path_to_experiment, f"checkpoint_{completed_steps}")

                if accelerator.is_main_process:
                    progress_bar.write(f"Saving Checkpoint to {path_to_checkpoint}")

                ### Make sure that all processes have caught up before saving checkpoint! ###
                accelerator.wait_for_everyone()

                ### Save checkpoint using only the main process ###
                if accelerator.is_main_process:
                    accelerator.save_state(output_dir=path_to_checkpoint)

                ### Delete Old Checkpoints ###
                if args.num_keep_checkpoints is not None:

                    if accelerator.is_main_process:

                        all_checkpoints = os.listdir(path_to_experiment)
                        all_checkpoints = sorted(all_checkpoints, key=lambda x: int(x.split(".")[0].split("_")[-1]))

                        if len(all_checkpoints) > args.num_keep_checkpoints:
                            checkpoints_to_delete = all_checkpoints[:-args.num_keep_checkpoints]

                            for checkpoint_to_delete in checkpoints_to_delete:
                                path_to_checkpoint_to_delete = os.path.join(path_to_experiment, checkpoint_to_delete)
                                if os.path.isdir(path_to_checkpoint_to_delete):
                                    shutil.rmtree(path_to_checkpoint_to_delete)

                ### Let all processes hang out while files are being deleted with the main process ###
                accelerator.wait_for_everyone()

            if completed_steps >= args.num_training_steps:
                train = False
                if accelerator.is_main_process:
                    progress_bar.write("Completed Training!!")
                break

            ### Iterate Progress Bar and Completed Steps ###
            completed_steps += 1
            progress_bar.update(1)

            ### Reset Topk Accuracy and Loss Accumulate For Next Accumulation ###
            accuracy = 0
            accumulate_loss = 0

accelerator.end_training()


pretrain.sh

In [ ]:
accelerate launch pretrain_roberta.py \
    --experiment_name "RoBERTa_Pretraining" \
    --working_directory "work_dir" \
    --hf_model_name "FacebookAI/roberta-base" \
    --path_to_prepped_data "/mnt/datadrive/data/prepped_data/roberta_data" \
    --context_length 512 \
    --masking_probability 0.15 \
    --num_workers 24 \
    --hidden_dropout_p 0.1 \
    --attention_dropout_p 0.1 \
    --num_transformer_blocks 12 \
    --num_attention_heads 12 \
    --embedding_dimension 768 \
    --mlp_ratio 4 \
    --max_grad_norm 1.0 \
    --layer_norm_eps 1e-5 \
    --initializer_range 0.02 \
    --per_gpu_batch_size 512 \
    --gradient_accumulation_steps 8 \
    --num_training_steps 300000 \
    --num_warmup_steps 20000 \
    --lr_scheduler linear \
    --logging_steps 1 \
    --evaluation_interval 2500 \
    --checkpoint_interval 2500 \
    --learning_rate 6e-4 \
    --weight_decay 0.01 \
    --adam_beta1 0.9 \
    --adam_beta2 0.98 \
    --adam_epsilon 1e-6 \
    --seed 42 \
    --log_wandb

In [ ]:
if __name__ == "__main__":
    from utils import RobertaConfig
